# Stage 5-2: Trainer와 Evaluator 구현

이 노트북은 `src/core/trainer.py`의 `Trainer`와 `src/core/evaluator.py`의 `Evaluator`를 실습한다.
`Trainer.fit`은 DataLoader를 받아 epoch 단위 학습 루프를 실행하고 loss/metric 로그를 반환한다.
`Evaluator.evaluate`는 test set에 대한 평균 loss와 metric을 집계한다.

In [ ]:
# sys.path 설정 — 프로젝트 루트를 모듈 검색 경로에 추가한다.
import sys, os
sys.path.insert(0, os.path.abspath("../.."))

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
from src.data.mnist import MnistDataset, get_task_spec
from src.data.dataloader import DataLoader
from src.models.mlp import MLP
from src.core.optimizers import SGD, Adam
from src.core.trainer import Trainer
from src.core.evaluator import Evaluator

## 1. task_spec과 dispatch 구조

`Trainer`와 `Evaluator`는 `task_spec` dict를 받아 내부에서 loss, gradient, metric 함수를 자동으로 선택한다.
클라이언트 코드에는 task 분기가 없다.

```
task 이름 → _TASK_FNS[task] → (loss_fn, grad_fn, metric_fn)
```

| task | loss_fn | grad_fn | metric_fn |
|---|---|---|---|
| `multiclass` | `cross_entropy` | `cross_entropy_grad` | `accuracy` |
| `binary` | `binary_cross_entropy` | `binary_cross_entropy_grad` | `binary_accuracy` |
| `regression` | `mse` | `mse_grad` | `r2_score` |

In [ ]:
# task_spec 구조 확인
for task in ["multiclass", "binary", "regression"]:
    spec = get_task_spec(task)
    print(f"[{task}] task_spec: {spec}")

## 2. Trainer 동작 확인

`Trainer.fit`은 한 epoch 학습 후 `{"loss", "metric", "num_samples"}`를 반환한다.
여러 epoch 실행은 클라이언트가 루프를 돌며 `fit`을 반복 호출한다.

In [ ]:
DATASET_DIR = "/mnt/d/datasets/mnist"
task = "multiclass"
task_spec = get_task_spec(task)

train_ds = MnistDataset("train", task, dataset_dir=DATASET_DIR)
test_ds = MnistDataset("test", task, dataset_dir=DATASET_DIR)
train_loader = DataLoader(train_ds, batch_size=128, shuffle=True)
test_loader = DataLoader(test_ds, batch_size=256, shuffle=False)

model = MLP(task=task, seed=42)
optimizer = SGD(model, lr=0.01)
trainer = Trainer(model, optimizer, task_spec)

train_log = trainer.fit(train_loader)
print("fit 반환값:", train_log)

In [ ]:
# 반환값 키 및 타입 검증
assert "loss" in train_log
assert "metric" in train_log
assert "num_samples" in train_log
assert train_log["num_samples"] == len(train_ds)
print(f"train loss={train_log['loss']:.4f}, metric={train_log['metric']:.4f}, samples={train_log['num_samples']}")

## 3. 배치 학습 한 스텝 상세

한 batch에서 파라미터 $\theta$를 업데이트하는 흐름:

```
forward → loss 계산 → gradient 계산 → backward → optimizer.step
```

샘플 수 가중 평균 집계:

$$\bar{L}_{\text{epoch}} = \frac{\sum_b L_b \cdot n_b}{\sum_b n_b}$$

마지막 배치가 `batch_size`보다 작아도 정확한 epoch 평균이 보장된다.

In [ ]:
# 가중 평균 집계 원리 — 배치별 loss * 샘플 수를 누적 후 전체로 나눈다.
from src.nn.losses import cross_entropy
from src.nn.metrics import accuracy

model_check = MLP(task="multiclass", seed=42)
small_loader = DataLoader(MnistDataset("train", "multiclass", dataset_dir=DATASET_DIR),
                          batch_size=64, shuffle=False)

total_loss, total_n = 0.0, 0
for i, (xb, yb) in enumerate(small_loader):
    logits = model_check.forward(xb)
    batch_loss = float(cross_entropy(logits, yb))
    n = len(xb)
    total_loss += batch_loss * n
    total_n += n
    if i == 2:
        break
print(f"3 배치 가중 평균 loss: {total_loss / total_n:.4f} (총 {total_n} 샘플)")

## 4. Evaluator 동작 확인

`Evaluator.evaluate`는 `model.eval()`로 전환 후 forward + loss/metric 집계만 수행한다.
backward를 호출하지 않으므로 파라미터가 변경되지 않는다.

In [ ]:
evaluator = Evaluator(model, task_spec)
params_before = [p.copy() for p in model.params]

test_log = evaluator.evaluate(test_loader)
print("evaluate 반환값:", test_log)

# 파라미터 불변 확인
unchanged = all(np.allclose(p, b) for p, b in zip(model.params, params_before))
assert unchanged, "evaluate는 파라미터를 변경하지 않아야 한다."
print("파라미터 불변 확인 통과")

In [ ]:
# num_samples 정확성 확인
assert test_log["num_samples"] == len(test_ds)
print(f"test loss={test_log['loss']:.4f}, metric={test_log['metric']:.4f}, samples={test_log['num_samples']}")

## 5. Trainer + Evaluator 통합 — 5 epoch 학습 곡선

매 epoch마다 `trainer.fit`과 `evaluator.evaluate`를 모두 호출하여
train/test loss와 metric을 동시에 기록한다.

In [ ]:
model_5ep = MLP(task="multiclass", seed=42)
opt_5ep = SGD(model_5ep, lr=0.01)
trainer_5ep = Trainer(model_5ep, opt_5ep, task_spec)
evaluator_5ep = Evaluator(model_5ep, task_spec)

logs = []
for epoch in range(1, 6):
    tr = trainer_5ep.fit(train_loader)
    te = evaluator_5ep.evaluate(test_loader)
    logs.append({"epoch": epoch, "train": tr, "test": te})
    print(f"Epoch {epoch} | train loss={tr['loss']:.4f} metric={tr['metric']:.4f}"
          f" | test loss={te['loss']:.4f} metric={te['metric']:.4f}")

In [ ]:
epochs = [log["epoch"] for log in logs]
train_losses = [log["train"]["loss"] for log in logs]
test_losses = [log["test"]["loss"] for log in logs]
train_metrics = [log["train"]["metric"] for log in logs]
test_metrics = [log["test"]["metric"] for log in logs]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(epochs, train_losses, marker="o", label="train")
ax1.plot(epochs, test_losses, marker="s", label="test")
ax1.set_xlabel("Epoch")
ax1.set_ylabel("Loss")
ax1.set_title("Loss (multiclass)")
ax1.legend()

ax2.plot(epochs, train_metrics, marker="o", label="train")
ax2.plot(epochs, test_metrics, marker="s", label="test")
ax2.set_xlabel("Epoch")
ax2.set_ylabel("Accuracy")
ax2.set_title("Accuracy (multiclass)")
ax2.legend()

plt.tight_layout()
plt.show()

## 6. 3 task 동작 확인

`Trainer`와 `Evaluator`는 동일한 인터페이스로 3 task를 처리한다.
클라이언트는 `task_spec`만 바꾸면 된다.

In [ ]:
for task in ["multiclass", "binary", "regression"]:
    ts = get_task_spec(task)
    tr_ds = MnistDataset("train", task, dataset_dir=DATASET_DIR)
    te_ds = MnistDataset("test", task, dataset_dir=DATASET_DIR)
    tr_loader = DataLoader(tr_ds, batch_size=256, shuffle=True)
    te_loader = DataLoader(te_ds, batch_size=256, shuffle=False)

    m = MLP(task=task, seed=42)
    opt = SGD(m, lr=0.01)
    tr_obj = Trainer(m, opt, ts)
    ev_obj = Evaluator(m, ts)

    tr_log = tr_obj.fit(tr_loader)
    te_log = ev_obj.evaluate(te_loader)

    assert "loss" in tr_log and "metric" in tr_log
    assert "loss" in te_log and "metric" in te_log
    print(f"[{task}] train: loss={tr_log['loss']:.4f}, metric={tr_log['metric']:.4f}"
          f" | test: loss={te_log['loss']:.4f}, metric={te_log['metric']:.4f}")

print("\n3 task 동작 검증 통과")

## 7. Trainer와 Evaluator 역할 비교

| 항목 | Trainer | Evaluator |
|---|---|---|
| 목적 | 파라미터 업데이트 | 모델 성능 측정 |
| gradient 계산 | `grad_fn` 포함 | 없음 |
| backward | `model.backward(grad)` 호출 | 호출하지 않음 |
| optimizer | `optimizer.step()` 호출 | 사용하지 않음 |
| model 모드 | `model.train()` 전환 | `model.eval()` 전환 |
| 반환값 | `loss`, `metric`, `num_samples` | `loss`, `metric`, `num_samples` |

**주요 설계 원칙**
- `_TASK_FNS` dict dispatch로 task별 함수를 생성 시 바인딩한다
- 샘플 수 가중 평균(`total_loss += loss * n`)으로 마지막 배치 크기 편향을 제거한다
- `Evaluator`는 backward가 없어 평가 중 파라미터가 변경되지 않는다